In [1]:
# 데이터 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df= pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [2]:
# 문자열 column 소문자 및 공백 _ 처리하기
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [3]:
# df 형태 type 확인하기
display(df.dtypes)
# pd.to_numeric(df.totalcharges) # column 내부에 결측치가 있을 경우 형 변환 X

df.totalcharges = pd.to_numeric(df.totalcharges,errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

# target 값 1/0으로 전환하기
df.churn = (df.churn =='yes').astype(int)
df.churn

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: churn, Length: 7043, dtype: int64

In [4]:
# train,valid,test 나누기
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df,test_size=0.2,random_state=1)
df_train, df_valid = train_test_split(df_full_train,test_size= 0.2,random_state=1)

df_train = df_train.reset_index(drop=True)
df_valid = df_valid.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

X_train = df_train.drop('churn',axis=1)
y_train = df_train['churn']

X_valid = df_valid.drop('churn',axis=1)
y_valid = df_valid['churn']

X_test = df_test.drop('churn',axis=1)
y_test = df_test['churn']

In [5]:
# 범주형 변수의 범위
df[categorical_columns].nunique()

customerid          7043
gender                 2
partner                2
dependents             2
phoneservice           2
multiplelines          3
internetservice        3
onlinesecurity         3
onlinebackup           3
deviceprotection       3
techsupport            3
streamingtv            3
streamingmovies        3
contract               3
paperlessbilling       2
paymentmethod          4
totalcharges        6531
churn                  2
dtype: int64

In [6]:
# group별 churn rate 보기
global_churn = df_full_train.churn.mean()
print(global_churn)

# global churn에서 churn female과 churn male이 크게 떨어져 있지 않으므로 유의미하지 않은 변수임
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()

# partner의 유무의 차이는 변화율이 눈에 띄게 있음 >> 유의미한 변수가 될 수 있음
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

0.26996805111821087


In [7]:
# mutal information : categorical variables의 중요도 보기 information theory 기반
from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

# 이 변수를 알면 정답을 맞히는 데 얼마나 도움이 되는가?"를 확률의 관점에서 엄밀하게 측정하는 지표
mi = df_full_train[categorical_columns].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

c:\Users\오소민\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)


churn               0.583227
customerid          0.583227
totalcharges        0.547955
contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

In [19]:
# 수치형 변수만 골라내기
numerical_columns = df.dtypes[df.dtypes != 'object'].index

**Categorical Value : Mutal Information** & **Numerical Value : Correlation**\

1. When r is positive, an increase in x will increase y.
2. When r is negative, an increase in x will decrease y.
3. When r is 0, a change in x does not affect y.

In [18]:
# 범주형 변수 같은 경우는 mutal information을 위주로 봄
# 상관관계 : A와 B가 같은 방향으로 변하는 경향이 있는가? -> direction과 상관 없이 본다면

df_full_train[numerical_columns].corrwith(df_full_train['churn']).abs()

seniorcitizen     0.141966
tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
churn             1.000000
dtype: float64

In [28]:
X_train.to_dict(orient='records')

[{'customerid': '7026-ymsbe',
  'gender': 'male',
  'seniorcitizen': 0,
  'partner': 'no',
  'dependents': 'no',
  'tenure': 30,
  'phoneservice': 'yes',
  'multiplelines': 'no',
  'internetservice': 'no',
  'onlinesecurity': 'no_internet_service',
  'onlinebackup': 'no_internet_service',
  'deviceprotection': 'no_internet_service',
  'techsupport': 'no_internet_service',
  'streamingtv': 'no_internet_service',
  'streamingmovies': 'no_internet_service',
  'contract': 'one_year',
  'paperlessbilling': 'no',
  'paymentmethod': 'mailed_check',
  'monthlycharges': 19.7,
  'totalcharges': 625.05},
 {'customerid': '0027-kwykw',
  'gender': 'female',
  'seniorcitizen': 0,
  'partner': 'yes',
  'dependents': 'yes',
  'tenure': 23,
  'phoneservice': 'yes',
  'multiplelines': 'yes',
  'internetservice': 'fiber_optic',
  'onlinesecurity': 'no',
  'onlinebackup': 'no',
  'deviceprotection': 'no',
  'techsupport': 'no',
  'streamingtv': 'yes',
  'streamingmovies': 'no',
  'contract': 'month-to-mon

In [21]:
# One Hot Encoding > One hot encoder 외에도 Dictvectorizer로 numerical이랑 categorical을 동시에 처리해주기도 함
from sklearn.feature_extraction import DictVectorizer
# One Hot Encoding > One hot encoder 외에도 DictVectorizer로 numerical이랑 categorical을 동시에 처리해주기도 함 it.

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(X_train.to_dict(orient='records'))
X_train

array([[0.00000e+00, 1.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        3.00000e+01, 6.25050e+02],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        2.30000e+01, 1.84995e+03],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        1.40000e+01, 1.34630e+03],
       ...,
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        2.00000e+00, 1.90050e+02],
       [0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 0.00000e+00,
        2.70000e+01, 7.61950e+02],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        9.00000e+00, 7.51650e+02]], shape=(4507, 4552))

In [ ]:
# One Hot Encoder
# X_train에 존재하는 범주형 컬럼만 선택
categorical_columns = X_train.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder

oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # sparse: 0을 제외한 값들을 저장 /  dense : 0을 포함한 모든 값들을 저장

X_train_encoded = oh.fit_transform(X_train[categorical_columns])
X_train_encoded

array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.]], shape=(4507, 4548))

In [ ]:
# 1. Logistic Regression 모델 학습
from sklearn.linear_model import LogisticRegression

# 모델 정의
model = LogisticRegression(solver='lbfgs', max_iter=1000)

# 모델 학습
model.fit(X_train, y_train)

# 2. 학습된 가중치 확인
print("절편:", model.intercept_[0])  # Bias (w0)
print("가중치:", model.coef_[0])     # 각 Feature의 가중치

# 3. Validation 예측
# predict(): 바로 0 또는 1로 예측
y_val_pred = model.predict(X_val)

# predict_proba(): 각 클래스에 속할 확률 반환
y_val_proba = model.predict_proba(X_val)

# 클래스 1(churn=1)에 속할 확률만 추출
y_val_pred_pro = model.predict_proba(X_val)[:, 1]

# 4. 임계값(Threshold) 0.5 적용
churn_decision = (y_val_pred_pro >= 0.5).astype(int)

# 5. 정확도(Accuracy) 계산
accuracy = (churn_decision == y_val).mean()

print("Validation Accuracy:", accuracy)